# 03 — Silver Transformations

**Week:** 5

Goal: Standardize raw Bronze data into clean Silver table(s).


# Week 5 – Silver Candidate Transformations

This notebook transforms the Week 4 Bronze tables into Silver Candidate tables.

Main steps:
1. Check Bronze tables
2. Record Bronze row counts
3. Standardize text fields
4. Convert required data types using TRY_CAST
5. Create Silver Candidate Delta tables
6. Validate row counts
7. Validate lineage and controlled reruns


In [0]:
display(spark.sql("""
SELECT
    current_catalog() AS current_catalog,
    current_schema() AS current_schema
"""))

## 1. Bronze Table Readiness

The following Bronze tables were created in Week 4:

- bronze_bookings
- bronze_guests
- bronze_rate_plans
- bronze_room_nights
- bronze_rooms

In [0]:
display(spark.sql("SHOW TABLES"))

## 2. Bronze Row Count Baseline

Record the row counts before performing Silver Candidate transformations.

In [0]:
display(spark.sql("""
SELECT 'bronze_bookings' AS table_name, COUNT(*) AS row_count
FROM bronze_bookings

UNION ALL

SELECT 'bronze_guests', COUNT(*)
FROM bronze_guests

UNION ALL

SELECT 'bronze_rate_plans', COUNT(*)
FROM bronze_rate_plans

UNION ALL

SELECT 'bronze_room_nights', COUNT(*)
FROM bronze_room_nights

UNION ALL

SELECT 'bronze_rooms', COUNT(*)
FROM bronze_rooms
"""))

## 3. Transform Bronze Bookings to Silver Candidate

The transformation standardizes text fields and safely converts date and timestamp fields.

No rows are intentionally removed or duplicated.

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver_bookings_candidate
USING DELTA
AS
SELECT
    TRIM(booking_id) AS booking_id,
    TRIM(property_id) AS property_id,
    TRIM(guest_id) AS guest_id,
    TRIM(rate_plan_id) AS rate_plan_id,
    TRIM(requested_room_type) AS requested_room_type,

    TRY_CAST(booking_date AS DATE) AS booking_date,
    TRY_CAST(arrival_date AS DATE) AS arrival_date,
    TRY_CAST(departure_date AS DATE) AS departure_date,

    UPPER(TRIM(booking_status)) AS booking_status,
    TRIM(market_segment) AS market_segment,
    TRIM(channel) AS channel,

    adults,
    children,
    rooms_booked,
    lead_time_days,

    nightly_rate,
    discount_amount,
    tax_amount,
    refund_amount,
    booked_amount,
    net_booking_value,

    TRY_CAST(cancellation_ts AS TIMESTAMP) AS cancellation_ts,
    TRY_CAST(checkin_ts AS TIMESTAMP) AS checkin_ts,
    TRY_CAST(checkout_ts AS TIMESTAMP) AS checkout_ts,

    ingestion_timestamp,
    source_file_name,
    source_system,
    batch_id

FROM bronze_bookings
""")

In [0]:
display(spark.sql("""
SELECT *
FROM silver_bookings_candidate
LIMIT 10
"""))

In [0]:
display(spark.sql("""
DESCRIBE silver_bookings_candidate
"""))

### Bookings Row Count Reconciliation

Compare Bronze and Silver Candidate row counts to confirm that no rows were lost or duplicated.

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM bronze_bookings) AS bronze_count,
    (SELECT COUNT(*) FROM silver_bookings_candidate) AS silver_candidate_count
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver_guests_candidate
USING DELTA
AS
SELECT
    TRIM(guest_id) AS guest_id,
    UPPER(TRIM(guest_segment)) AS guest_segment,
    UPPER(TRIM(origin_region)) AS origin_region,
    UPPER(TRIM(loyalty_tier)) AS loyalty_tier,
    
    repeat_guest_flag,

    ingestion_timestamp,
    source_file_name,
    source_system,
    batch_id

FROM bronze_guests
""")


In [0]:
display(spark.sql("""
SELECT *
FROM silver_guests_candidate
LIMIT 10
"""))

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM bronze_guests) AS bronze_count,
    (SELECT COUNT(*) FROM silver_guests_candidate) AS silver_candidate_count
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver_rate_plans_candidate
USING DELTA
AS
SELECT
    TRIM(rate_plan_id) AS rate_plan_id,
    TRIM(property_id) AS property_id,
    UPPER(TRIM(room_type)) AS room_type,
    UPPER(TRIM(channel)) AS channel,

    effective_from,
    effective_to,

    nightly_rate,
    UPPER(TRIM(meal_plan)) AS meal_plan,
    TRIM(cancellation_policy) AS cancellation_policy,
    discount_pct,

    ingestion_timestamp,
    source_file_name,
    source_system,
    batch_id

FROM bronze_rate_plans
""")

In [0]:
display(spark.sql("""
SELECT *
FROM silver_rate_plans_candidate
LIMIT 10
"""))

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM bronze_rate_plans) AS bronze_count,
    (SELECT COUNT(*) FROM silver_rate_plans_candidate) AS silver_candidate_count
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver_room_nights_candidate
USING DELTA
AS
SELECT
    TRIM(room_night_id) AS room_night_id,
    TRIM(booking_id) AS booking_id,
    TRIM(property_id) AS property_id,
    TRIM(room_id) AS room_id,
    UPPER(TRIM(room_type)) AS room_type,

    stay_date,

    TRIM(rate_plan_id) AS rate_plan_id,

    occupied_flag,
    available_flag,
    revenue_eligible_flag,
    complimentary_flag,
    out_of_service_flag,

    recognized_room_revenue,

    ingestion_timestamp,
    source_file_name,
    source_system,
    batch_id

FROM bronze_room_nights
""")

In [0]:
display(spark.sql("""
SELECT *
FROM silver_room_nights_candidate
LIMIT 10
"""))

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM bronze_room_nights) AS bronze_count,
    (SELECT COUNT(*) FROM silver_room_nights_candidate) AS silver_candidate_count
"""))

In [0]:
spark.sql("""
CREATE OR REPLACE TABLE silver_rooms_candidate
USING DELTA
AS
SELECT
    TRY_CAST(active_from AS DATE) AS active_from,
    TRY_CAST(active_to AS DATE) AS active_to,

    capacity,
    floor,
    out_of_service_flag,

    TRIM(property_id) AS property_id,
    TRIM(property_name) AS property_name,
    TRIM(room_id) AS room_id,
    UPPER(TRIM(room_type)) AS room_type,
    UPPER(TRIM(wing)) AS wing,

    ingestion_timestamp,
    source_file_name,
    source_system,
    batch_id

FROM bronze_rooms
""")

In [0]:
display(spark.sql("""
SELECT *
FROM silver_rooms_candidate
LIMIT 10
"""))

In [0]:
display(spark.sql("""
SELECT
    (SELECT COUNT(*) FROM bronze_rooms) AS bronze_count,
    (SELECT COUNT(*) FROM silver_rooms_candidate) AS silver_candidate_count
"""))

In [0]:
display(spark.sql("""
SELECT 'bookings' AS table_name,
       (SELECT COUNT(*) FROM bronze_bookings) AS bronze_count,
       (SELECT COUNT(*) FROM silver_bookings_candidate) AS silver_count

UNION ALL

SELECT 'guests',
       (SELECT COUNT(*) FROM bronze_guests),
       (SELECT COUNT(*) FROM silver_guests_candidate)

UNION ALL

SELECT 'rate_plans',
       (SELECT COUNT(*) FROM bronze_rate_plans),
       (SELECT COUNT(*) FROM silver_rate_plans_candidate)

UNION ALL

SELECT 'room_nights',
       (SELECT COUNT(*) FROM bronze_room_nights),
       (SELECT COUNT(*) FROM silver_room_nights_candidate)

UNION ALL

SELECT 'rooms',
       (SELECT COUNT(*) FROM bronze_rooms),
       (SELECT COUNT(*) FROM silver_rooms_candidate)
"""))

In [0]:
display(spark.sql("""
SHOW TABLES
"""))

In [0]:
display(spark.sql("""
SELECT
    'Week 5 Silver Candidate Transformation Completed Successfully' AS status
"""))

In [0]:
bronze_df = spark.table("bronze_bookings")

silver_df = bronze_df.selectExpr(
    "TRIM(booking_id) AS booking_id",
    "TRIM(property_id) AS property_id",
    "TRIM(guest_id) AS guest_id",
    "TRIM(rate_plan_id) AS rate_plan_id",
    "TRIM(requested_room_type) AS requested_room_type",

    "TRY_CAST(booking_date AS DATE) AS booking_date",
    "TRY_CAST(arrival_date AS DATE) AS arrival_date",
    "TRY_CAST(departure_date AS DATE) AS departure_date",

    "UPPER(TRIM(booking_status)) AS booking_status",
    "TRIM(market_segment) AS market_segment",
    "TRIM(channel) AS channel",

    "adults",
    "children",
    "rooms_booked",
    "lead_time_days",

    "nightly_rate",
    "discount_amount",
    "tax_amount",
    "refund_amount"
)

display(silver_df.limit(10))

In [0]:
silver_table = "silver_standardized_events"
silver_df.write.mode("overwrite").saveAsTable(silver_table)

print("Silver table created:", silver_table)
print("Silver row count:", spark.table(silver_table).count())


In [0]:
%sql

SELECT 
    booking_status,
    COUNT(*) AS record_count,
    SUM(nightly_rate) AS total_nightly_rate

FROM silver_bookings_candidate

GROUP BY booking_status

ORDER BY record_count DESC


## Evidence to Capture

- One raw-to-Silver mapping example
- Silver schema screenshot
- Silver row count screenshot


# Week 5 – Silver Candidate Transformations

This notebook transforms the Week 4 Bronze tables into Silver Candidate tables.

Main steps:
1. Check Bronze tables
2. Record Bronze row counts
3. Standardize text fields
4. Convert required data types using TRY_CAST
5. Create Silver Candidate Delta tables
6. Validate row counts
7. Validate lineage and controlled reruns


In [0]:
display(spark.sql("SHOW TABLES"))

In [0]:
bronze_df = spark.table("bronze_bookings")

bronze_df.printSchema()

In [0]:
display(spark.sql("""
SELECT 'bookings' AS table_name,
       (SELECT COUNT(*) FROM bronze_bookings) AS bronze_count,
       (SELECT COUNT(*) FROM silver_bookings_candidate) AS silver_count

UNION ALL

SELECT 'guests',
       (SELECT COUNT(*) FROM bronze_guests),
       (SELECT COUNT(*) FROM silver_guests_candidate)

UNION ALL

SELECT 'rate_plans',
       (SELECT COUNT(*) FROM bronze_rate_plans),
       (SELECT COUNT(*) FROM silver_rate_plans_candidate)

UNION ALL

SELECT 'room_nights',
       (SELECT COUNT(*) FROM bronze_room_nights),
       (SELECT COUNT(*) FROM silver_room_nights_candidate)

UNION ALL

SELECT 'rooms',
       (SELECT COUNT(*) FROM bronze_rooms),
       (SELECT COUNT(*) FROM silver_rooms_candidate)
"""))